# 10. 결과 해석 — 전체 11단계 중 10단계

작업하며 주고받은 글(내가 셀마다 적은 칸과 클로드의 답변)은 빼고 코드 · 결과 · 결정만 남긴 판이다. 진행 방식은 README 「진행 방식」에 있다.

**이 단계에서 한 일과 정한 것** — 9단계의 차이를 LNG 배 몇 척으로 바꿔 봐도 달마다 평소 범위 안이라는 판정은 같았다. 석탄 발전량이 많았던 까닭을 찾아봤지만, LNG와 석탄의 연료비단가 차이로는 설명되지 않았고 까닭은 찾지 못했다. 이 프로젝트가 본 것(LNG 발전량 하나)과 보지 못한 것(재고 · 들여온 양 같은 것)을 인과지도와 저량 · 유량 그림으로 그렸다.

### 말 뜻 — 표를 읽을 때 쓰는 말

| 말 | 뜻 |
|---|---|
| 실제 | 그해(주로 2026년)에 실제로 발전한 양 · 실제 값입니다. 전력거래소 자료 그대로입니다 |
| 모델 예측 | 2023 ~ 2025년을 배운 모델이 그때의 날씨 · 시각 · 날 종류를 보고 예측한 양입니다. 9단계까지는 「평소라면」이라고 불렀습니다 |
| 실제 − 모델 예측 | 발전량 표의 숫자입니다. + 면 실제가 모델 예측보다 많았다, − 면 적었다는 뜻입니다 |
| 평소 해 | 2023 · 2024 · 2025년입니다. 「평소」는 이 세 해만 가리킵니다 |
| 평소 범위 | 평소 해에 모델 예측이 한 달 평균으로 가장 크게 틀린 크기(±2,177 MWh)입니다. 모델이 예측해 본 2024 · 2025년으로 냈습니다. 2026년의 차이가 이 안이면 「달랐다고 말할 근거가 없다」로 읽습니다 |
| 세 해 평균 | 2023 ~ 2025년 같은 달 값의 평균입니다. 모델 없이 계산한 것으로, 10단계 가격 표(2-가 · 2-라)에서만 씁니다 |

앞 단계 글과, 이 노트북에서 말을 통일하기 전에 쓴 글에서 발전량을 두고 쓴 「평소라면」 · 「평소보다 많았다 · 적었다」 · 「실제 − 평소」는 「모델 예측」 · 「모델 예측보다 많았다 · 적었다」 · 「실제 − 모델 예측」으로 읽습니다.

In [148]:
# 10단계 결과 해석 · 준비 ①
import os, glob                                # 폴더와 파일 경로를 다루는 기본 모듈 · 파일 이름을 한꺼번에 찾는 모듈
import numpy as np                             # 숫자 계산
import pandas as pd                            # 표(데이터프레임)를 다루는 라이브러리
from IPython.display import display            # 표를 글자가 아니라 격자로 그린다
pd.set_option("display.max_columns", 50)       # 열이 많아도 생략하지 않고 다 보이게

ROOT = os.getcwd()                             # 지금 폴더(보통 notebooks/)에서 시작해서
while not os.path.isdir(os.path.join(ROOT, "data")) and ROOT != os.path.dirname(ROOT):
    ROOT = os.path.dirname(ROOT)               # data 폴더가 보일 때까지 위로 = 프로젝트 루트
RAW = os.path.join(ROOT, "data", "raw")        # 원자료 폴더 — 2단계에서 받아 둔 것
HOT = os.path.join(ROOT, "data", "hot")        # 5단계가 만들어 둔 파일이 있는 폴더

print("프로젝트 루트:", ROOT)                    # 경로가 맞는지만 확인한다

프로젝트 루트: /Users/kimchangju/vs코드 /개인 프로젝트/lng-project


In [149]:
# 10단계 결과 해석 · 준비 ②
# ── 9단계 준비 ② · 1-다와 같은 계산 — 평소 해에 그래디언트 부스팅이 달마다 틀린 양과, 그 가운데 가장 큰 크기(평소 범위) ──
from sklearn.ensemble import HistGradientBoostingRegressor                   # 8단계에서 고른 그래디언트 부스팅
tbl = pd.read_csv(os.path.join(HOT, "features_2023_2024_2025.csv"), encoding="utf-8-sig")   # 5단계 「저장」 셀이 만든 표 — 6 ~ 9단계와 같은 파일
tbl["ts"] = pd.to_datetime(tbl["ts"])                                         # 글자로 저장된 시각을 다시 시각 자료형으로
tbl = tbl.set_index("ts").sort_index()                                        # 시각을 줄 이름으로 두고 시간 순서대로 놓는다
전체 = tbl.dropna(subset=["3시간 전 기온"])                                    # 3시간 전 기온이 빈 9시각은 뺀다 — 6 ~ 9단계와 같다
y = 전체["lng_mwh"]                                                           # 맞힐 값 — 그 시각의 LNG 발전량(MWh)
X = 전체[["hour24", "month", "icsr", "ta", "dow", "is_holiday", "명절"]]       # 트리 계열 7열 — 6단계에서 정하고 8단계에서 확정한 입력

def 그래디언트부스팅():                                                         # 8단계에서 확정한 설정 그대로
    return HistGradientBoostingRegressor(early_stopping=False, random_state=0)

평소 = {}                                                                     # 맞힌 해 → 시각마다 실제 − 예측(+ 면 실제가 예측보다 많았다)
for 맞힌해, 배운해 in [(2024, [2023]), (2025, [2023, 2024])]:                  # 2023 → 2024 · 2023 · 2024 → 2025 — 9단계와 같은 두 번
    배울곳 = 전체["year"].isin(배운해); 맞힐곳 = 전체["year"] == 맞힌해
    평소[맞힌해] = y[맞힐곳] - 그래디언트부스팅().fit(X[배울곳], y[배울곳]).predict(X[맞힐곳])
달평균 = pd.concat([잔차.groupby(잔차.index.month).mean() for 잔차 in 평소.values()])   # 스물세 달(2024년 8월은 자료가 없다)마다 시각마다 틀린 양의 평균
평소범위 = 달평균.abs().max()                                                    # + · −를 떼고 가장 큰 달 = 평소 범위(9단계 물음 1의 기준)
같다 = round(평소범위) == 2177                                                  # 9단계 1-다 [표 2]의 「그 달의 값」 크기
print(f"평소 범위를 다시 냈다 — 9단계 1-다의 2,177 MWh와 {'같다' if 같다 else '다르다(scikit-learn 판을 확인해 주세요)'}")

평소 범위를 다시 냈다 — 9단계 1-다의 2,177 MWh와 같다


In [150]:
# 10단계 결과 해석 · 준비 ③
# ── 9단계 준비 ③ · 2-가와 같은 계산 — 2026년 1 ~ 6월 「실제 − 모델 예측」을 달마다 한 시각 평균으로 ──
파일 = [os.path.join(RAW, "asos", "asos_108_202512.csv")] + sorted(glob.glob(os.path.join(RAW, "asos", "asos_108_2026*.csv")))   # 2025년 12월은 1월 1일 0 ~ 2시의 3시간 전 기온에만 쓴다
기상26 = pd.concat([pd.read_csv(f, encoding="utf-8-sig", usecols=["tm", "ta", "icsr"]) for f in 파일], ignore_index=True)   # 서울 ASOS — 시각 · 기온 · 일사만 읽는다
기상26["ts"] = pd.to_datetime(기상26["tm"])                                          # '2026-01-01 00:00' 글자를 시각으로
기상26 = 기상26.set_index("ts").sort_index()                                         # 시각을 줄 이름으로 두고 시간 순서대로
기상26["3시간 전 기온"] = 기상26["ta"].shift(freq="3h").reindex(기상26.index)           # 3시간 전 기온 — 5단계 물음 2와 같은 열
입력26 = 기상26.loc["2026-01-01 00:00":"2026-06-30 23:00"].drop(columns="tm").copy()   # 비교 기간 1 ~ 6월만 남긴다
입력26["icsr"] = 입력26["icsr"].fillna(0)                                              # 해가 없을 때 비는 일사는 0 — 3단계와 같다
입력26["year"] = 입력26.index.year                                                     # 2026
입력26["month"] = 입력26.index.month                                                   # 1 ~ 6월
입력26["hour24"] = 입력26.index.hour                                                   # 0 ~ 23시
입력26["dow"] = 입력26.index.dayofweek                                                 # 요일 0=월 … 6=일
휴일 = pd.read_csv(os.path.join(RAW, "holidays_kr.csv"), encoding="utf-8-sig")          # 2단계 `02_수집_공휴일`이 받아 둔 파일
날짜 = pd.Series(입력26.index.date, index=입력26.index)                                 # 시각마다 그 날짜
입력26["is_holiday"] = 날짜.isin(set(pd.to_datetime(휴일["date"]).dt.date)).astype(int)   # 공휴일이면 1 — 5단계와 같다
명절날짜 = set(pd.to_datetime(휴일.loc[휴일["name"].str.contains("설날|추석"), "date"]).dt.date)   # 이름에 설날 · 추석이 든 날
입력26["명절"] = 날짜.isin(명절날짜).astype(int)                                          # 설 · 추석 연휴면 1 — 5단계와 같다
요일 = 입력26["dow"]
입력26["날 종류"] = (요일.map({0: "월", 1: "화~금", 2: "화~금", 3: "화~금", 4: "화~금", 5: "토", 6: "일"})   # 5단계 「저장」 셀과 같은 여섯 가지
                    .where(~((입력26["is_holiday"] == 1) & (요일 <= 4) & (입력26["명절"] == 0)), "공휴일")    # 평일에 든 명절 아닌 공휴일
                    .where(입력26["명절"] == 0, "명절"))                                                  # 명절인 날은 요일과 상관없이
ta, ta3 = 입력26["ta"], 입력26["3시간 전 기온"]                                         # 지금 기온 · 3시간 전 기온
입력26["16도보다 추운 만큼"] = (16 - ta).clip(lower=0)                                  # 선형회귀(준비 ④)가 쓰는 열 — 5단계와 같다
입력26["24도보다 더운 만큼"] = (ta - 24).clip(lower=0)
입력26["3시간 전 16도보다 추운 만큼"] = (16 - ta3).clip(lower=0)
입력26["3시간 전 24도보다 더운 만큼"] = (ta3 - 24).clip(lower=0)

실제 = []                                                                              # 2026년 LNG 실제 값 — 연료 여덟이 24시간 다 온 날만(3단계와 같은 기준)
for f in sorted(glob.glob(os.path.join(RAW, "api", "2026*.csv"))):                     # 2026년 하루치 파일을 날짜 순서대로
    하루 = pd.to_datetime(os.path.basename(f)[:8])                                     # 파일 이름의 날짜(20260101)
    if 하루.month > 6 or os.path.getsize(f) < 100: continue                            # 비교 기간 1 ~ 6월만 · 0행으로 온 날은 건너뛴다
    t = pd.read_csv(f, encoding="utf-8-sig")                                           # 그날 파일을 읽는다
    if len(t) != 192: continue                                                         # 연료 여덟 × 24시간 = 192행이 아니면 쓰지 않는다 — 5월 2 ~ 17일이 여기 걸린다
    t = t[t["fuelTpCd"] == "LNG"]                                                      # LNG 줄만
    실제.append(pd.Series(t["amgo"].values, index=하루 + pd.to_timedelta(t["tradeNo"].astype(int) - 1, unit="h")))   # 1 ~ 24시를 0 ~ 23시 시각으로 — 3단계와 같다
실제 = pd.concat(실제).sort_index()                                                     # 시각마다 실제 LNG 발전량(MWh)
평소라면 = pd.Series(그래디언트부스팅().fit(X, y).predict(입력26.loc[실제.index, X.columns]), index=실제.index)   # 2023 ~ 2025년 세 해를 배운 모델 — 6단계 물음 5 · 9단계 2-가와 같다
차이 = 실제 - 평소라면                                                                   # + 면 실제가 모델 예측보다 많았다
달 = 차이.groupby(차이.index.month)                                                      # 달마다 묶는다
달차이 = pd.DataFrame({"비교한 날": 달.apply(lambda s: s.index.normalize().nunique()),     # 그 달에 실제 LNG가 다 온 날 수
                       "한 시각 평균": 달.mean()})                                        # 9단계 2-가 표의 「실제 − 모델 예측」과 같은 값
print("2026년 1 ~ 6월 「실제 − 모델 예측」을 다시 냈다 — 비교한 날: " + " · ".join(f"{m}월 {n}일" for m, n in 달차이["비교한 날"].items()))

2026년 1 ~ 6월 「실제 − 모델 예측」을 다시 냈다 — 비교한 날: 1월 31일 · 2월 28일 · 3월 31일 · 4월 30일 · 5월 15일 · 6월 30일


In [151]:
# 10단계 결과 해석 · 준비 ④
# ── 9단계 1-사 · 2-나와 같은 계산 — 선형회귀로 「실제 − 평소라면」을 수요 · 원전 · 석탄 · 그 밖 넷으로 나눈다 ──
from sklearn.linear_model import LinearRegression                                    # 선형회귀 — 7단계와 같다
def 선형열(d):                                                                        # 7단계 준비 ②의 「선형열」과 같은 67열
    시각열 = pd.get_dummies(d["hour24"].astype(str), prefix="시", drop_first=True).astype(float)   # 1~23시 0/1 열 — 0시가 기준
    월열 = pd.get_dummies(d["month"].astype(str), prefix="월", drop_first=True).astype(float)      # 2~12월 0/1 열 — 1월이 기준
    날열 = pd.get_dummies(d["날 종류"], prefix="날").astype(float).drop(columns="날_월")            # 화~금·토·일·공휴일·명절 — 월요일이 기준
    쉬는날 = d["날 종류"].isin(["토", "일", "공휴일", "명절"]).astype(float)                         # 쉬는 날이면 1
    쉬는날시간대 = (시각열 - 1 / 24).mul(쉬는날, axis=0)                                            # 5단계 물음 7 — 쉬는 날 × 시간대
    쉬는날시간대.columns = [f"쉬는 날×{c[2:]}시" for c in 시각열.columns]
    기상 = pd.DataFrame({"16도보다 추운 만큼": d["16도보다 추운 만큼"], "24도보다 더운 만큼": d["24도보다 더운 만큼"],
                         "3시간 동안 바뀐 만큼 (추운 쪽)": d["16도보다 추운 만큼"] - d["3시간 전 16도보다 추운 만큼"],
                         "3시간 동안 바뀐 만큼 (더운 쪽)": d["24도보다 더운 만큼"] - d["3시간 전 24도보다 더운 만큼"],
                         "일사": d["icsr"]})
    return pd.concat([시각열, 월열, 날열, 기상, 쉬는날시간대], axis=1)
선형X = 선형열(전체)                                                                  # 준비 ②의 「전체」로 만든다

석탄종류 = ["유연탄", "무연탄"]                                                          # 석탄 = 유연탄 + 무연탄
def 연간(해):                                                                            # 전력거래소 연간 파일 한 해 — 시각마다 「LNG 밖 모든 발전원의 합」과 「석탄」
    이름 = [f for f in os.listdir(os.path.join(RAW, "kpx")) if f"{해}1231" in f][0]       # 파일 이름에 20251231 같은 기준일이 있다
    d = pd.read_csv(os.path.join(RAW, "kpx", 이름), encoding="utf-8-sig")
    d.columns = ["date", "hour", "fuel", "cap_mw", "mwh"]                               # 해마다 열 이름이 달라서 순서로 통일한다(3단계와 같다)
    d["fuel"] = d["fuel"].str.strip()
    d = d[d["fuel"] != "LNG"]                                                            # LNG 줄은 읽자마자 버린다 — LNG는 준비 ②의 값을 쓴다
    ts = pd.to_datetime(d["date"]) + pd.to_timedelta(d["hour"].astype(int) - 1, unit="h")   # 1 ~ 24시를 0 ~ 23시 시각으로
    석 = d["fuel"].isin(석탄종류)
    return d["mwh"].groupby(ts).sum(), d["mwh"][석].groupby(ts[석]).sum()
def API(해, 끝달):                                                                       # API 하루치 파일 — 연료 여덟이 24시간 다 온 날만, 발전원마다 한 열
    조각 = []
    for f in sorted(glob.glob(os.path.join(RAW, "api", f"{해}*.csv"))):
        하루 = pd.to_datetime(os.path.basename(f)[:8])
        if 하루.month > 끝달 or os.path.getsize(f) < 100: continue                       # 그 달까지만 · 0행으로 온 날은 건너뛴다
        t = pd.read_csv(f, encoding="utf-8-sig")
        if len(t) != 192: continue                                                       # 연료 여덟 × 24시간이 아닌 날은 쓰지 않는다
        t["ts"] = 하루 + pd.to_timedelta(t["tradeNo"].astype(int) - 1, unit="h")
        조각.append(t.pivot_table(index="ts", columns="fuelTpCd", values="amgo", aggfunc="sum"))
    return pd.concat(조각).sort_index().drop(columns="LNG")                              # LNG 열은 버린다 — LNG는 준비 ② · ③의 값을 쓴다
a24, a26 = API(2024, 12), API(2026, 6)
밖23, 석23 = 연간(2023)
밖25, 석25 = 연간(2025)
LNG밖 = pd.concat([밖23, a24.sum(axis=1), 밖25]).sort_index().reindex(전체.index)        # 2023 ~ 2025년 시각마다 LNG를 뺀 모든 발전원의 합
석탄 = pd.concat([석23, a24[석탄종류].sum(axis=1), 석25]).sort_index().reindex(전체.index)
원전 = 전체["nuke_mwh"]                                                                  # 7단계 3-거와 같은 원전
대상 = {"수요": y + LNG밖, "원전": 원전, "석탄": 석탄, "그 밖": LNG밖 - 원전 - 석탄}         # 수요 = LNG + LNG 밖 모든 발전원 · 그 밖 = 수력 · 양수 · 유류 · 태양광 · 풍력 같은 것
a26 = a26.reindex(실제.index)                                                            # 준비 ③과 같은 시각
대상26 = {"수요": 실제 + a26.sum(axis=1), "원전": a26["원자력"], "석탄": a26[석탄종류].sum(axis=1),
          "그 밖": a26.sum(axis=1) - a26["원자력"] - a26[석탄종류].sum(axis=1)}
assert all(v.notna().all() for v in list(대상.values()) + list(대상26.values())), "빈 시각이 있다"

X선26 = 선형열(입력26).reindex(columns=선형X.columns, fill_value=0).loc[실제.index]       # 2026년에는 7 ~ 12월 열이 없어서 0으로 채운다
배운곳 = 전체["year"].isin([2023, 2024, 2025])                                           # 그래디언트 부스팅과 같은 세 해를 배운다
평소라면값 = lambda 값: LinearRegression().fit(선형X[배운곳], 값[배운곳]).predict(X선26)   # 같은 67열로 그 값의 「평소라면」을 맞힌다
몫 = pd.DataFrame({"수요": 대상26["수요"] - 평소라면값(대상["수요"])}, index=실제.index)    # 수요가 평소보다 많으면 + — LNG가 늘었다
for k in ["원전", "석탄", "그 밖"]:
    몫[k] = -(대상26[k] - 평소라면값(대상[k]))                                           # 발전원이 평소보다 적으면 + — 그만큼 LNG가 늘었다
몫["선형회귀"] = 몫[["수요", "원전", "석탄", "그 밖"]].sum(axis=1)                        # 넷을 더하면 선형회귀가 본 LNG 차이
어긋남 = (몫["선형회귀"] - (실제 - 평소라면값(y))).abs().max()                             # 선형회귀가 LNG를 바로 맞힌 차이와 넷의 합 — 0이어야 한다
달몫 = 몫.groupby(몫.index.month).mean()                                                 # 달마다 한 시각 평균 — 9단계 2-나 [표 1]과 같은 값
del 실제, 평소라면, 차이, 몫                                                             # 2026년 시각마다의 LNG 값은 여기서 버린다 — 10단계에는 달마다 한 시각 평균만 남긴다
print(f"선형회귀로 넷으로 나눴다 — 넷의 합과 선형회귀 LNG 차이가 가장 크게 어긋난 시각도 {어긋남:.6f} MWh (0이어야 한다)")

선형회귀로 넷으로 나눴다 — 넷의 합과 선형회귀 LNG 차이가 가장 크게 어긋난 시각도 0.000000 MWh (0이어야 한다)


# 물음 1 — 9단계의 차이를 LNG 몇 톤 · 몇 척으로 바꾸면 얼마인가

> **결정** — 평소 범위는 배 약 3 ~ 4척(LNG 약 23 ~ 25만 톤, 30일 기준)이고, 2026년 1 ~ 6월은 가장 크게 달랐던 6월도 약 2 ~ 3척(약 17 ~ 18만 톤) 줄어든 정도라 달마다 그 안이었다

## 1-나 — 가정 셋(발열량 · 효율 · 배 한 척 물량)의 값과 출처

**고르는 규칙** (값을 찾기 전에 정한 것 — 넷째 · 다섯째 줄은 10-28에 이 셀을 본 뒤에 바꿨습니다)

**표의 숫자**

- 한국전력통계 효율은 실제 발전소들의 평균입니다. 평소와 달랐던 만큼의 전기는 평균보다 효율이 낮은 발전소에서 나와서, 실제 배 척 수는 「척이 많이 나오는 끝」보다 많을 수 있습니다(판단).
- 한국전력통계 효율 줄은 한전 자회사 복합화력 것으로 보입니다. 표가 「한전 자회사 계」와 「기타사」로 나뉘고, 기타사는 발전 형식으로 나뉘어 있지 않기 때문입니다(판단). 우리 자료에는 민간 발전사도 들어 있습니다.

In [152]:
# 10단계 결과 해석 · 1-나
# ── 가정 셋 — 값 · 출처 · 확인 정도(확인 = 원본을 직접 봄) ──
발열량 = {"총발열량": 54.7, "순발열량": 49.4}                                       # LNG 1 kg에서 나오는 열(MJ) — 1톤당 GJ와 같은 숫자다 · 에너지법 시행규칙 별표(2022.11.21 개정)
통계 = {2023: 45.38218890, 2024: 45.29090762, 2025: 45.69903185}                    # 한전 자회사 복합화력의 송전단 열효율(%) — EPSIS 「화력발전소 열효율」, 출처 한국전력통계 책자
효율 = np.mean(list(통계.values())) / 100                                            # 2023 ~ 2025년 세 해 평균 — 효율은 한국 발전소의 값 하나만 쓴다
한척 = {"125,000 m³": 57_000, "135,000 m³": 62_000, "170,000 m³": 72_000}           # 배 한 척에 싣는 LNG(톤)

표1 = pd.DataFrame([                                                                 # [표 1] 가정마다 값 · 출처 · 확인 정도
    ["발열량 · 총발열량 (물이 증발할 때의 열까지 넣은 값)", "54.7 MJ/kg", "에너지법 시행규칙 별표 (2022.11.21 개정)", "확인"],
    ["발열량 · 순발열량 (그 열을 뺀 값)", "49.4 MJ/kg", "같은 표", "확인"],
    ["효율 · 한국전력통계 2023 ~ 2025년 평균", f"{효율 * 100:.2f}%", "EPSIS 「화력발전소 열효율」 복합화력 송전단 열 (한전 자회사로 보임)", "값과 송전단은 확인 · 한전 자회사인지는 판단 · 총/순 기준은 확인 불가"],
    ["배 한 척 · 125,000 m³", "57,000톤", "한국가스공사 누리집", "확인"],
    ["배 한 척 · 135,000 m³", "62,000톤", "한국가스공사 누리집", "확인"],
    ["배 한 척 · 170,000 m³ 만재", "72,000톤", "Riviera", "확인"],
], columns=["가정", "값", "출처", "확인 정도"]).set_index("가정")

def 톤GWh(효, 발):                                                                    # 1 GWh를 만드는 데 드는 LNG(톤)
    return 1000 * 3.6 / 효 / 발                                                      # 1,000 MWh × 3.6 GJ/MWh ÷ 효율 ÷ 발열량(GJ/톤)
끝 = {"척이 적게 나오는 끝": ("총발열량", 한척["170,000 m³"]),                         # 통계 효율이 총발열량 기준이라면 · 배가 가장 클 때
      "척이 많이 나오는 끝": ("순발열량", 한척["125,000 m³"])}                         # 통계 효율이 순발열량 기준이라면 · 배가 가장 작을 때
줄들 = []                                                                              # [표 2]의 줄을 모을 목록
for k, (발, 척) in 끝.items():
    t = 톤GWh(효율, 발열량[발])                                                        # 1 GWh에 드는 LNG(톤)
    줄들.append({"끝": k, "효율": f"{효율 * 100:.2f}%", "발열량": f"{발} {발열량[발]} MJ/kg",
                 "1 GWh에 드는 LNG (톤)": f"{t:,.1f}", "배 한 척 (톤)": f"{척:,}", "1척이 몇 GWh": f"{척 / t:,.1f}"})
두끝 = pd.DataFrame(줄들).set_index("끝")
톤값 = {k: 톤GWh(효율, 발열량[발]) for k, (발, 척) in 끝.items()}                       # 1 GWh에 드는 LNG(톤) — 척으로 바꿀 때 쓴다
척값 = {k: 척 / 톤값[k] for k, (발, 척) in 끝.items()}                                  # 1척이 몇 GWh — 척으로 바꿀 때 쓴다

print("[표 1] 가정 셋 — 확인 = 원본을 직접 봄")
display(표1)
print("[표 2] 두 끝 — 1 GWh에 드는 LNG = 1,000 MWh × 3.6 GJ/MWh ÷ 효율 ÷ 발열량, 1척이 몇 GWh = 배 한 척 ÷ 1 GWh에 드는 LNG")
print("       평소와 달랐던 만큼의 전기는 평균보다 효율이 낮은 발전소에서 나와서, 실제 배 척 수는 「척이 많이 나오는 끝」보다 많을 수 있다(판단)")
display(두끝)

[표 1] 가정 셋 — 확인 = 원본을 직접 봄


,값,출처,확인 정도
가정,,,
발열량 · 총발열량 (물이 증발할 때의 열까지 넣은 값),54.7 MJ/kg,에너지법 시행규칙 별표 (2022.11.21 개정),확인
발열량 · 순발열량 (그 열을 뺀 값),49.4 MJ/kg,같은 표,확인
효율 · 한국전력통계 2023 ~ 2025년 평균,45.46%,EPSIS 「화력발전소 열효율」 복합화력 송전단 열 (한전 자회사로 보임),값과 송전단은 확인 · 한전 자회사인지는 판단 · 총/순 기준은 확인 불가
"배 한 척 · 125,000 m³","57,000톤",한국가스공사 누리집,확인
"배 한 척 · 135,000 m³","62,000톤",한국가스공사 누리집,확인
"배 한 척 · 170,000 m³ 만재","72,000톤",Riviera,확인


[표 2] 두 끝 — 1 GWh에 드는 LNG = 1,000 MWh × 3.6 GJ/MWh ÷ 효율 ÷ 발열량, 1척이 몇 GWh = 배 한 척 ÷ 1 GWh에 드는 LNG
       평소와 달랐던 만큼의 전기는 평균보다 효율이 낮은 발전소에서 나와서, 실제 배 척 수는 「척이 많이 나오는 끝」보다 많을 수 있다(판단)


,효율,발열량,1 GWh에 드는 LNG (톤),배 한 척 (톤),1척이 몇 GWh
끝,,,,,
척이 적게 나오는 끝,45.46%,총발열량 54.7 MJ/kg,144.8,"72,000",497.3
척이 많이 나오는 끝,45.46%,순발열량 49.4 MJ/kg,160.3,"57,000",355.6


## 1-다 — 9단계의 차이와 평소 범위를 배 몇 척으로

**표의 숫자**

- [표 1] 달마다 9단계의 차이 「실제 − 모델 예측」(한 시각 평균 MWh — + 면 실제가 모델 예측보다 많았다)을 30일 기준 GWh로 바꾸고, 1-나의 두 끝으로 LNG 만 톤과 척을 냅니다. 만 톤은 GWh × 「1 GWh에 드는 LNG」(1-나 [표 2]) ÷ 10,000, 척은 GWh ÷ 「1척이 몇 GWh」입니다. 맨 아래 줄은 평소 범위(평소 해에 그래디언트 부스팅이 가장 크게 틀린 달의 크기)를 같은 방법으로 바꾼 것입니다.
- 30일 기준은 한 시각 평균에 720시각을 곱한 양입니다. 그 달의 실제 양이 아니고, 달마다 날 수가 달라도(2월 28일 · 5월 15일) 평소 범위와 같은 기준으로 비교하려고 이렇게 바꿉니다. 5월은 15일 평균을 30일로 늘린 값입니다.
- 톤과 척은 발전에 쓴 LNG를 바꾼 양이고, 들여온 LNG의 양이나 배의 수가 아닙니다. 들여오는 양은 계약 물량 · 저장 재고 · 도시가스 수요로도 정해집니다.
- 「평소 범위 안인가」는 9단계와 같은 판정입니다. 차이와 범위에 같은 수를 곱해서, 척으로 바꿔도 바뀌지 않습니다.
- [표 2] 9단계 2-나의 나눈 표를 같은 30일 기준 GWh로 바꾼 것입니다. 맨 위 줄은 [표 1]의 차이를 GWh로 적은 것이고, 둘째 줄은 선형회귀가 본 차이(아래 네 줄의 합)입니다. 네 줄은 선형회귀로 나눈 몫이라 척으로 바꾸지 않습니다.
- 모델끼리 비교하는 표가 아닙니다.

- 결론 문장에 쓸 숫자는 [표 1] 맨 아래 「평소 범위」 줄의 두 끝입니다 — 「평소 해에도 가장 크게 틀린 달은 a ~ b척(30일 기준)만큼 어긋났다」.
- 달마다 척 수는 평소 범위 줄과 비교해 읽습니다. 범위 안의 차이를 「n척 더 썼다」로 읽지 않습니다.
- 실제 척 수는 「척이 많이 나오는 끝」 쪽이거나 그보다 조금 많을 수 있습니다 — 평소와 달랐던 만큼은 평균보다 효율이 낮은 발전소에서 나와서입니다(1-나, 판단).
- [표 2]는 「왜 평소 범위 안이었나」를 GWh로 보는 표입니다. 숫자는 9단계 2-나와 같은 값을 30일 기준으로 바꾼 것이라, 새로 알게 되는 것은 단위뿐입니다.

In [153]:
# 10단계 결과 해석 · 1-다
# ── 9단계의 차이와 평소 범위를 척으로 — 둘 다 30일 기준(한 시각 평균 × 720시각)으로, 1-나의 같은 값을 곱한다 ──
def GWh30(mwh): return mwh * 720 / 1000                                              # 한 시각 평균(MWh) × 720시각 ÷ 1,000 = 30일 기준(GWh)
척적게, 척많이 = 1 / 척값["척이 적게 나오는 끝"], 1 / 척값["척이 많이 나오는 끝"]        # 1 GWh당 척 — 1-나 [표 2]의 「1척이 몇 GWh」를 뒤집은 값
줄 = []
for m, r in 달차이.iterrows():                                                        # 달마다
    g = GWh30(r["한 시각 평균"])
    줄.append({"달": f"{m}월", "비교한 날": f"{int(r['비교한 날'])}일",
               "실제 − 모델 예측 · 한 시각 평균 (MWh)": f"{r['한 시각 평균']:+,.0f}",
               "30일 기준 (GWh)": f"{g:+,.0f}",
               "만 톤 · 적게 나오는 끝": f"{g * 톤값['척이 적게 나오는 끝'] / 1e4:+.1f}",      # GWh × 1 GWh에 드는 LNG(톤) ÷ 10,000
               "만 톤 · 많이 나오는 끝": f"{g * 톤값['척이 많이 나오는 끝'] / 1e4:+.1f}",
               "척 · 적게 나오는 끝": f"{g * 척적게:+.1f}", "척 · 많이 나오는 끝": f"{g * 척많이:+.1f}",
               "평소 범위 안인가": "안" if abs(r["한 시각 평균"]) <= 평소범위 else "밖"})   # 9단계와 같은 판정 — 같은 수를 곱해도 바뀌지 않는다
g = GWh30(평소범위)
줄.append({"달": "평소 범위", "비교한 날": "—", "실제 − 모델 예측 · 한 시각 평균 (MWh)": f"±{평소범위:,.0f}",
           "30일 기준 (GWh)": f"±{g:,.0f}",
           "만 톤 · 적게 나오는 끝": f"±{g * 톤값['척이 적게 나오는 끝'] / 1e4:.1f}", "만 톤 · 많이 나오는 끝": f"±{g * 톤값['척이 많이 나오는 끝'] / 1e4:.1f}",
           "척 · 적게 나오는 끝": f"±{g * 척적게:.1f}", "척 · 많이 나오는 끝": f"±{g * 척많이:.1f}", "평소 범위 안인가": "—"})
표1 = pd.DataFrame(줄).set_index("달")

이름 = {"그래디언트 부스팅": "그래디언트 부스팅 — LNG가 모델 예측과 달랐던 양 (위 표와 같은 값)",
        "선형회귀": "선형회귀 — 아래 넷을 더한 값",
        "수요": "수요 줄 — + 면 수요가 모델 예측보다 많아서 LNG가 늘었다",
        "원전": "원전 줄 — + 면 원전 발전량이 모델 예측보다 적어서 LNG가 늘었다",
        "석탄": "석탄 줄 — + 면 석탄 발전량이 모델 예측보다 적어서 LNG가 늘었다",
        "그 밖": "그 밖 줄 — + 면 그 밖 발전량이 모델 예측보다 적어서 LNG가 늘었다"}
표2 = 달몫.assign(**{"그래디언트 부스팅": 달차이["한 시각 평균"]})[list(이름)].T.rename(index=이름)   # 9단계 2-나 [표 1]과 같은 줄
표2 = 표2.apply(lambda c: GWh30(c).map("{:+,.0f}".format))                                  # 30일 기준 GWh로
표2.columns = [f"{m}월" for m in 표2.columns]

print("[표 1] 2026년 달마다 LNG가 모델 예측과 달랐던 양을 톤과 척으로 — 30일 기준은 그 달의 실제 양이 아니라 한 시각 평균 × 720시각")
print("       5월은 15일 평균을 30일로 늘린 값 · 톤과 척은 발전에 쓴 LNG를 바꾼 양이고, 들여온 LNG의 양이나 배의 수가 아니다")
display(표1)
print("[표 2] 같은 30일 기준 GWh로 본 9단계 2-나의 나눈 표 — 네 줄은 선형회귀로 나눈 몫이라 척으로 바꾸지 않는다")
print(f"       척으로 읽고 싶으면 1척 ≈ {1 / 척많이:,.0f} ~ {1 / 척적게:,.0f} GWh (1-나의 두 끝)")
display(표2)

[표 1] 2026년 달마다 LNG가 모델 예측과 달랐던 양을 톤과 척으로 — 30일 기준은 그 달의 실제 양이 아니라 한 시각 평균 × 720시각
       5월은 15일 평균을 30일로 늘린 값 · 톤과 척은 발전에 쓴 LNG를 바꾼 양이고, 들여온 LNG의 양이나 배의 수가 아니다


,비교한 날,실제 − 모델 예측 · 한 시각 평균 (MWh),30일 기준 (GWh),만 톤 · 적게 나오는 끝,만 톤 · 많이 나오는 끝,척 · 적게 나오는 끝,척 · 많이 나오는 끝,평소 범위 안인가
달,,,,,,,,
1월,31일,"+1,096",+789,+11.4,+12.7,+1.6,+2.2,안
2월,28일,"+1,446","+1,041",+15.1,+16.7,+2.1,+2.9,안
3월,31일,"+1,211",+872,+12.6,+14.0,+1.8,+2.5,안
4월,30일,+257,+185,+2.7,+3.0,+0.4,+0.5,안
5월,15일,+38,+27,+0.4,+0.4,+0.1,+0.1,안
6월,30일,"-1,595","-1,148",-16.6,-18.4,-2.3,-3.2,안
평소 범위,—,"±2,177","±1,568",±22.7,±25.1,±3.2,±4.4,—


[표 2] 같은 30일 기준 GWh로 본 9단계 2-나의 나눈 표 — 네 줄은 선형회귀로 나눈 몫이라 척으로 바꾸지 않는다
       척으로 읽고 싶으면 1척 ≈ 356 ~ 497 GWh (1-나의 두 끝)


,1월,2월,3월,4월,5월,6월
그래디언트 부스팅 — LNG가 모델 예측과 달랐던 양 (위 표와 같은 값),+789,"+1,041",+872,+185,+27,"-1,148"
선형회귀 — 아래 넷을 더한 값,+809,"+1,008",+746,+235,+194,"-1,235"
수요 줄 — + 면 수요가 모델 예측보다 많아서 LNG가 늘었다,-419,-774,"-1,214",-582,+6,"-1,154"
원전 줄 — + 면 원전 발전량이 모델 예측보다 적어서 LNG가 늘었다,"+1,551","+2,625","+2,749","+1,059","+1,618",+500
석탄 줄 — + 면 석탄 발전량이 모델 예측보다 적어서 LNG가 늘었다,-5,-561,-804,-238,"-2,039","-1,072"
그 밖 줄 — + 면 그 밖 발전량이 모델 예측보다 적어서 LNG가 늘었다,-319,-282,+15,-4,+608,+491


# 물음 2 — 석탄이 2026년에 모델 예측보다 많았던 까닭은 무엇인가

> **결정** — 석탄 발전량이 모델 예측보다 많았던 것은 값 차이로 설명되지 않는다. 석탄과 LNG의 발전량이 각각 모델 예측보다 많았던 양을 합친 것 가운데 석탄의 비율은 1월 1%에서 5월 91%로 늘었고, 시기가 맞는 까닭 셋은 데이터로 구별할 수 없다

## 2-가

**표의 숫자** — 달마다 LNG 연료비단가에서 석탄(유연탄) 연료비단가를 뺀 값, 곧 「LNG 값 − 석탄 값」(원/kWh)입니다. 연료비단가는 발전소가 전기 1 kWh를 만드는 데 쓴 연료의 값입니다. 전력거래소 규정은 이 값의 바탕인 열량단가를 연료비 · 환경비 · 배출권 비용을 합한 값으로 정하고 있어서, 배출권 비용도 들어 있는 것으로 봅니다. 유연탄은 석탄 발전소가 쓰는 석탄입니다. 무연탄도 석탄이지만 우리 자료에서 석탄 발전량의 1%가 안 돼서(2023년 0.94%, 2025년 0.73% — 전력거래소 연간 파일) 넣지 않았습니다. + 면 LNG가 석탄보다 비쌌다는 뜻이고, 맨 오른쪽 「2026년 − 세 해 평균」이 + 면 2026년에 LNG가 더 비쌌다는 뜻입니다. 세 해 평균은 같은 달 2023 ~ 2025년 값의 평균입니다(모델 없음). 모델끼리 비교하는 표가 아닙니다.

- 「2026년 − 세 해 평균」이 1 ~ 6월에 + 로 크면, LNG가 평소 해보다 더 비싸서 석탄을 더 썼다는 것이 석탄이 많았던 까닭 후보가 됩니다. 0 근처면 가격으로는 설명되지 않습니다.
- 석탄 발전량은 2월부터 모델 예측보다 많았고 봉쇄는 3월이라, 2월에 이미 + 였는지를 같이 봅니다.

In [154]:
# 10단계 결과 해석 · 2-가
# ── EPSIS 「연료비용」의 연료비단가(원/kWh)로, 달마다 LNG가 유연탄보다 얼마나 비쌌나를 해마다 나란히 ──
파일 = os.path.join(RAW, "HOME_전력거래_연료비용.csv")
머리 = pd.read_csv(파일, encoding="cp949", header=None, nrows=3)              # 위 세 줄 = 값 이름 · 연료 이름 · 단위 (한글 윈도 글자로 적힌 파일)
assert 머리.iloc[0, 11] == "연료비단가" and 머리.iloc[1, 12] == "유연탄" and 머리.iloc[1, 15] == "LNG"   # 쓸 두 칸이 맞는 칸인지 확인
원본 = pd.read_csv(파일, encoding="cp949", header=None, skiprows=3)           # 넷째 줄부터가 달마다의 값
가격 = pd.DataFrame({"해": 원본[0].str[:4].astype(int),                        # 「2026/09」의 앞 네 글자 = 해
                     "달": 원본[0].str[5:7].astype(int),                       # 뒤 두 글자 = 달
                     "유연탄": 원본[12].astype(float),                         # 연료비단가 · 유연탄 (원/kWh)
                     "LNG": 원본[15].astype(float)})                           # 연료비단가 · LNG (원/kWh)
가격 = 가격[가격["달"] <= 6]                                                    # 9단계에서 비교한 1 ~ 6월만 본다
가격["차이"] = 가격["LNG"] - 가격["유연탄"]                                     # + 면 LNG가 유연탄보다 비쌌다

표 = 가격.pivot(index="달", columns="해", values="차이")                       # 달 × 해 표
세해평균 = 표[[2023, 2024, 2025]].mean(axis=1)                                 # 같은 달 2023 ~ 2025년의 평균(모델 없음)
표1 = pd.DataFrame({"2023년": 표[2023], "2024년": 표[2024], "2025년": 표[2025],
                    "세 해 평균": 세해평균, "2026년": 표[2026],
                    "2026년 − 세 해 평균 (+ 면 2026년에 LNG가 더 비쌌다)": 표[2026] - 세해평균})
표1.index = [f"{m}월" for m in 표1.index]                                       # 1 → 「1월」

print("[표 1] 달마다 LNG 값 − 석탄(유연탄) 값, 연료비단가 (원/kWh) — + 면 LNG가 석탄보다 비쌌다")
print("       연료비단가 = 발전소가 전기 1 kWh를 만드는 데 쓴 연료의 값 (EPSIS 「연료비용」)")
print()
display(표1.apply(lambda 열: 열.map(lambda v: f"{v:+.1f}")))                   # 소수 한 자리, + · − 를 붙여서

[표 1] 달마다 LNG 값 − 석탄(유연탄) 값, 연료비단가 (원/kWh) — + 면 LNG가 석탄보다 비쌌다
       연료비단가 = 발전소가 전기 1 kWh를 만드는 데 쓴 연료의 값 (EPSIS 「연료비용」)



,2023년,2024년,2025년,세 해 평균,2026년,2026년 − 세 해 평균 (+ 면 2026년에 LNG가 더 비쌌다)
1월,+132.2,+92.8,+62.2,+95.8,+47.0,-48.8
2월,+145.9,+78.5,+54.8,+93.1,+47.5,-45.6
3월,+113.5,+72.6,+49.7,+78.6,+39.1,-39.5
4월,+67.7,+59.1,+52.9,+59.9,+45.3,-14.6
5월,+54.9,+54.7,+52.2,+54.0,+52.5,-1.5
6월,+53.2,+52.3,+47.8,+51.1,+52.4,+1.3


## 2-나

**표의 숫자** — 모든 칸의 + · − 는 「실제 − 모델 예측」입니다. **+ 면 그 발전원의 실제 발전량이 모델 예측보다 많았다, − 면 적었다**는 뜻입니다. 모델 예측은 2023 ~ 2025년을 배운 선형회귀가 예측한 값입니다(9단계 2-나와 같다) — 2023 ~ 2025년을 배운 선형회귀가 날씨 · 시각 · 날 종류가 같을 때 맞힌 그 발전원의 발전량입니다. 30일 기준 GWh(한 시각 평균 × 720시각)로 적었고, 5월은 15일 평균입니다. 「석탄이 대신 만든 비율」은 석탄이 많았던 양을 원전이 적었던 양으로 나눈 값으로, 100%면 원전이 적었던 만큼 석탄이 많았다는 뜻입니다. 모델끼리 비교하는 표가 아닙니다. 1-다 [표 2]는 LNG가 늘어난 쪽을 + 로 적은 표라서, 원전 · 석탄의 부호가 이 표와 반대입니다.

- 「석탄이 대신 만든 비율」이 달마다 비슷하게 크면(달마다 절반이 넘으면), 원전이 적었던 만큼을 석탄이 먼저 대신 만들었다는 것이 석탄이 많았던 까닭 후보가 됩니다.
- 달마다 크게 다르면 원전만으로는 설명되지 않습니다.

In [155]:
# 10단계 결과 해석 · 2-나
# ── 2026년 달마다 원전 · 석탄 발전량이 모델 예측과 얼마나 달랐나 — 실제 − 모델 예측, 30일 기준 GWh (+ 면 실제가 모델 예측보다 많았다, − 면 적었다) ──
GWh = lambda mwh: mwh * 720 / 1000                                              # 한 시각 평균(MWh) × 720시각 ÷ 1,000 = 30일 기준(GWh)
원전차이 = -GWh(달몫["원전"])                                                     # 준비 ④의 달몫은 LNG가 늘어난 쪽이 + 라서 부호를 바꾼다 → 실제 − 모델 예측
석탄차이 = -GWh(달몫["석탄"])                                                     # 석탄도 같다 → 실제 − 모델 예측
대신비율 = 석탄차이 / -원전차이 * 100                                             # 석탄이 더 만든 양 ÷ 원전이 덜 만든 양 — 100% = 원전이 덜 만든 양만큼 석탄이 더 만들었다
표1 = pd.DataFrame({"원전 (실제 − 모델 예측, GWh)": 원전차이.map(lambda v: f"{v:+,.0f}"),
                    "석탄 (실제 − 모델 예측, GWh)": 석탄차이.map(lambda v: f"{v:+,.0f}"),
                    "석탄이 대신 만든 비율 = 석탄이 더 만든 양 ÷ 원전이 덜 만든 양": 대신비율.map(lambda v: f"{v:.0f}%")})
표1.index = [f"{m}월" for m in 표1.index]                                         # 1 → 「1월」

print("[표 1] 2026년 달마다 원전 · 석탄 발전량 — 실제 − 모델 예측, 30일 기준 GWh · + 면 실제가 모델 예측보다 많았다, − 면 적었다")
print("       모델 예측 = 2023 ~ 2025년을 배운 선형회귀가 그 날씨 · 시각 · 날 종류를 보고 예측한 발전량 · 5월은 15일 평균")
print()
display(표1)

[표 1] 2026년 달마다 원전 · 석탄 발전량 — 실제 − 모델 예측, 30일 기준 GWh · + 면 실제가 모델 예측보다 많았다, − 면 적었다
       모델 예측 = 2023 ~ 2025년을 배운 선형회귀가 그 날씨 · 시각 · 날 종류를 보고 예측한 발전량 · 5월은 15일 평균



,"원전 (실제 − 모델 예측, GWh)","석탄 (실제 − 모델 예측, GWh)",석탄이 대신 만든 비율 = 석탄이 더 만든 양 ÷ 원전이 덜 만든 양
1월,"-1,551",+5,0%
2월,"-2,625",+561,21%
3월,"-2,749",+804,29%
4월,"-1,059",+238,22%
5월,"-1,618","+2,039",126%
6월,-500,"+1,072",214%


## 2-라

**표의 숫자** — 연료비단가(발전소가 전기 1 kWh를 만드는 데 쓴 연료의 값, 원/kWh)가 그해 2월보다 얼마나 달라졌나입니다. + 면 2월보다 비싸졌다는 뜻입니다. 2월은 봉쇄(2월 28일 공격 · 3월 4일 봉쇄 선언 — 확인, 9단계 4-가 1 · 3번) 전 마지막 달이라 기준으로 삼았습니다. 세 해 평균은 같은 달 2023 ~ 2025년 값의 평균이고(모델 없음), 맨 오른쪽 「2026년 − 세 해 평균」이 + 면 2026년에 평소 해보다 더 비싸졌다는 뜻입니다. 해마다 값의 높이가 달라서, 높이가 아니라 2월에서 달라진 만큼을 해마다 나란히 놓았습니다. 9단계 4-나처럼 사건 앞뒤의 변화를 평소 해와 나란히 놓는 방식입니다. 모델끼리 비교하는 표가 아닙니다.

**값이 늦게 따라오는 까닭(외부 자료)** — 시장에서 거래하는 LNG의 연료단가는 두 달 전에 들어온 연료의 값으로 매기고([에너지경제신문 2020-10-22](https://m.ekn.kr/view.php?key=529698) — 기사 하나), 한국에 들여오는 LNG 값은 유가를 3 ~ 4개월쯤 늦게 따라간다고 합니다([메디게이트뉴스(아시아경제) 2020-08-06](https://medigatenews.com/stand/2661) — 기사 하나). 둘 다 값을 매기는 방식에 대한 설명이라 2026년 일을 적은 것은 아닙니다(확인 정도: 기사 하나씩).

In [156]:
# 10단계 결과 해석 · 2-라
# ── 봉쇄 전 마지막 달(2월)보다 연료비단가가 얼마나 달라졌나 — LNG · 석탄(유연탄)을 해마다 나란히 놓고 평소 해와 비교 ──
파일 = os.path.join(RAW, "HOME_전력거래_연료비용.csv")                         # 2-가와 같은 파일
머리 = pd.read_csv(파일, encoding="cp949", header=None, nrows=3)               # 위 세 줄 = 값 이름 · 연료 이름 · 단위
assert 머리.iloc[0, 11] == "연료비단가" and 머리.iloc[1, 12] == "유연탄" and 머리.iloc[1, 15] == "LNG"   # 쓸 두 칸이 맞는지 확인
원본 = pd.read_csv(파일, encoding="cp949", header=None, skiprows=3)            # 넷째 줄부터가 달마다의 값
연료값 = pd.DataFrame({"해": 원본[0].str[:4].astype(int),                       # 「2026/09」의 앞 네 글자 = 해
                       "달": 원본[0].str[5:7].astype(int),                      # 뒤 두 글자 = 달
                       "석탄": 원본[12].astype(float),                          # 연료비단가 · 유연탄 (원/kWh)
                       "LNG": 원본[15].astype(float)})                          # 연료비단가 · LNG (원/kWh)
연료값 = 연료값[연료값["달"] <= 6]                                               # 1 ~ 6월만

def 이월보다(연료):                                                               # 그달 값 − 그해 2월 값을 해마다 나란히
    표 = 연료값.pivot(index="달", columns="해", values=연료)                      # 달 × 해
    차 = (표 - 표.loc[2]).loc[[3, 4, 5, 6]]                                      # 해마다 2월을 0으로 놓고, 봉쇄 뒤 3 ~ 6월만
    세해평균 = 차[[2023, 2024, 2025]].mean(axis=1)                               # 같은 달 2023 ~ 2025년의 평균(모델 없음)
    나온 = pd.DataFrame({"2023년": 차[2023], "2024년": 차[2024], "2025년": 차[2025],
                         "세 해 평균": 세해평균, "2026년": 차[2026],
                         "2026년 − 세 해 평균 (+ 면 2026년에 더 비싸졌다)": 차[2026] - 세해평균})
    나온.index = [f"{m}월" for m in 나온.index]                                   # 3 → 「3월」
    return 나온.apply(lambda 열: 열.map(lambda v: f"{v:+.1f}"))                  # 소수 한 자리, + · − 를 붙여서

print("[표 1] LNG 연료비단가가 그해 2월보다 얼마나 달라졌나 (원/kWh) — + 면 2월보다 비싸졌다")
print()
display(이월보다("LNG"))
print()
print("[표 2] 석탄(유연탄) 연료비단가가 그해 2월보다 얼마나 달라졌나 (원/kWh) — + 면 2월보다 비싸졌다")
print()
display(이월보다("석탄"))

[표 1] LNG 연료비단가가 그해 2월보다 얼마나 달라졌나 (원/kWh) — + 면 2월보다 비싸졌다



,2023년,2024년,2025년,세 해 평균,2026년,2026년 − 세 해 평균 (+ 면 2026년에 더 비싸졌다)
3월,-38.7,-1.2,-3.1,-14.3,-3.9,+10.4
4월,-88.0,-13.4,+2.7,-32.9,+1.6,+34.5
5월,-102.3,-16.7,-1.8,-40.3,+8.6,+48.9
6월,-107.5,-19.3,-5.5,-44.1,+17.4,+61.5



[표 2] 석탄(유연탄) 연료비단가가 그해 2월보다 얼마나 달라졌나 (원/kWh) — + 면 2월보다 비싸졌다



,2023년,2024년,2025년,세 해 평균,2026년,2026년 − 세 해 평균 (+ 면 2026년에 더 비싸졌다)
3월,-6.3,+4.8,+2.0,+0.1,+4.5,+4.3
4월,-9.7,+6.0,+4.6,+0.3,+3.8,+3.5
5월,-11.3,+7.1,+0.7,-1.2,+3.6,+4.7
6월,-14.7,+6.9,+1.5,-2.1,+12.5,+14.6


## 2-마

**표의 숫자** — 모든 칸은 「실제 − 모델 예측」(+ 면 실제가 모델 예측보다 많았다, − 면 적었다)이고, 30일 기준 GWh입니다. 모델 예측은 2023 ~ 2025년을 배운 선형회귀가 예측한 값입니다(9단계 2-나와 같다). 「그 밖」은 원전 · 석탄 · LNG를 뺀 발전원으로, 대부분 태양광 · 수력 · 풍력처럼 연료비가 거의 없는 발전원이고 양수 · 유류도 조금 들어 있습니다(유류는 LNG보다 비싸서 엄밀하게는 「석탄보다 싼」에 들지 않지만 양이 적습니다 — 판단). 「둘을 더한 값」이 − 면 석탄보다 싼 발전원이 그만큼 비었다는 뜻이고, 「석탄이 대신 만든 비율」은 석탄이 늘어난 양을 덜 만든 양으로 나눈 값입니다(100%면 다 대신 만들었다). 덜 만든 양 가운데 석탄이 대신 만들지 않은 나머지는 LNG가 더 만들었거나 수요가 모델 예측보다 적어서 필요 없었습니다 — 그래서 LNG 칸을 같이 놓았습니다. 숫자는 1-다 [표 2]의 원전 · 그 밖 · 석탄 줄과 같은 값을 「실제 − 모델 예측」으로 바꿔 적은 것이라, 1-다 [표 2]와 부호가 반대입니다. 모델끼리 비교하는 표가 아닙니다.

In [157]:
# 10단계 결과 해석 · 2-마
# ── 석탄보다 싼 발전원(원전 · 그 밖)이 모델 예측보다 덜 만든 양을 석탄이 얼마나 대신 만들었나 — 실제 − 모델 예측, 30일 기준 GWh ──
GWh = lambda mwh: mwh * 720 / 1000                                              # 한 시각 평균(MWh) × 720시각 ÷ 1,000 = 30일 기준(GWh)
원전차이 = -GWh(달몫["원전"])                                                     # 준비 ④의 달몫은 LNG가 늘어난 쪽이 + 라서 부호를 바꾼다 → 실제 − 모델 예측
그밖차이 = -GWh(달몫["그 밖"])                                                    # 그 밖(태양광 · 수력 · 풍력 같은 것과 양수 · 유류)도 같다 → 실제 − 모델 예측
덜만든양 = 원전차이 + 그밖차이                                                    # 둘을 더한 값 — − 면 석탄보다 싼 발전원이 모델 예측보다 그만큼 덜 만들었다
석탄차이 = -GWh(달몫["석탄"])                                                     # 석탄 → 실제 − 모델 예측
LNG차이 = GWh(달몫["선형회귀"])                                                   # 선형회귀가 본 LNG — 이미 + 면 LNG가 많았다(실제 − 모델 예측)
대신비율 = 석탄차이 / -덜만든양 * 100                                             # 석탄이 더 만든 양 ÷ 원전과 그 밖이 덜 만든 양 — 100% = 덜 만든 양을 석탄이 다 대신 만들었다
쉼표 = lambda v: f"{v:+,.0f}"                                                     # 천 단위 쉼표, + · − 를 붙여서
표1 = pd.DataFrame({"원전": 원전차이.map(쉼표), "그 밖": 그밖차이.map(쉼표),
                    "둘을 더한 값 (− 면 원전과 그 밖이 모델 예측보다 덜 만들었다)": 덜만든양.map(쉼표),
                    "석탄": 석탄차이.map(쉼표),
                    "석탄이 대신 만든 비율 = 석탄 ÷ 원전과 그 밖이 덜 만든 양": 대신비율.map(lambda v: f"{v:.0f}%"),
                    "LNG (선형회귀)": LNG차이.map(쉼표)})
표1.index = [f"{m}월" for m in 표1.index]                                         # 1 → 「1월」

print("[표 1] 2026년 달마다 발전원별 발전량 — 실제 − 모델 예측, 30일 기준 GWh · + 면 실제가 모델 예측보다 많았다, − 면 적었다 (5월은 15일 평균)")
print("       모델 예측 = 9단계 2-나와 같은 선형회귀의 예측 · 원전과 그 밖이 덜 만든 양 가운데 석탄이 대신 만들지 않은 나머지는, LNG가 더 만들었거나 수요가 모델 예측보다 적어서 필요 없었다")
print()
display(표1)

[표 1] 2026년 달마다 발전원별 발전량 — 실제 − 모델 예측, 30일 기준 GWh · + 면 실제가 모델 예측보다 많았다, − 면 적었다 (5월은 15일 평균)
       모델 예측 = 9단계 2-나와 같은 선형회귀의 예측 · 원전과 그 밖이 덜 만든 양 가운데 석탄이 대신 만들지 않은 나머지는, LNG가 더 만들었거나 수요가 모델 예측보다 적어서 필요 없었다



,원전,그 밖,둘을 더한 값 (− 면 원전과 그 밖이 모델 예측보다 덜 만들었다),석탄,석탄이 대신 만든 비율 = 석탄 ÷ 원전과 그 밖이 덜 만든 양,LNG (선형회귀)
1월,"-1,551",+319,"-1,232",+5,0%,+809
2월,"-2,625",+282,"-2,343",+561,24%,"+1,008"
3월,"-2,749",-15,"-2,764",+804,29%,+746
4월,"-1,059",+4,"-1,055",+238,23%,+235
5월,"-1,618",-608,"-2,226","+2,039",92%,+194
6월,-500,-491,-991,"+1,072",108%,"-1,235"


## 2-바

**표의 숫자** — 모든 칸은 「실제 − 모델 예측」(+ 면 실제가 모델 예측보다 많았다, − 면 적었다)이고, 30일 기준 GWh입니다(5월은 15일 평균). 모델 예측은 2-마와 같은 선형회귀(2023 ~ 2025년을 배움)가 그 발전원 하나만 맞힌 값입니다. 넷을 더한 값은 2-마의 「그 밖」 칸과 같습니다(표 아래 확인 줄). 모델끼리 비교하는 표가 아닙니다.

**넷으로 맞춘 방법** — 2026년 자료(API)는 그 밖을 수력 · 양수 · 유류 · 신재생 · 기타 넷으로 줍니다. 2023 · 2025년 연간 파일은 연료를 더 잘게 주어서 넷에 맞춰 묶었습니다 — 수력은 「수력」, 양수는 「양수」, 유류는 「중유 + 경유」, 나머지(태양광 · 풍력 · 바이오 · 연료전지 · 소수력 · IGCC · 부생가스 같은 것)는 신재생 · 기타입니다. 2024년은 두 자료가 겹쳐서 이렇게 묶는 것이 맞는지 봤습니다 — 수력은 두 자료가 같았고(소수력은 들어 있지 않다), IGCC는 API에서 신재생 · 기타에 들어 있었습니다. 2023년 연간 파일에는 소수력 줄이 따로 없어서 수력에 들어 있을 수 있습니다. 그렇다면 수력 칸이 30일에 약 17 GWh 낮게, 신재생 · 기타 칸이 그만큼 높게 나옵니다 — 2024년 연간 파일의 소수력이 30일에 약 52 GWh이고, 모델이 배운 세 해 가운데 2023년에만 수력에 들어 있으니 모델 예측에 그 3분의 1쯤이 들어간다고 본 어림입니다.

- 5 · 6월에 「넷을 더한 값」의 대부분이 양수 칸이면 — 「석탄보다 싼 발전원이 덜 만든 양을 석탄이 대신 만들었다」는 2-마의 읽기를 고쳐야 합니다.
- 대부분이 수력이나 신재생 · 기타 칸이면 — 2-마의 읽기는 그대로입니다. 다만 신재생 · 기타 가운데 무엇이 덜 만들었는지는 이 표로 알 수 없습니다.

In [158]:
# 10단계 결과 해석 · 2-바
# ── 「그 밖」을 넷(수력 · 양수 · 유류 · 신재생 · 기타)으로 나눠 본다 — 실제 − 모델 예측, 30일 기준 GWh ──
넷이름 = ["수력", "양수", "유류", "신재생 · 기타"]                                   # 2026년 자료(API)가 그 밖을 주는 넷
바꿈 = {"수력": "수력", "양수": "양수", "유전": "유류", "신재생·기타": "신재생 · 기타"}   # API의 연료 이름 → 이 표의 이름
def 연간넷(해):                                                                       # 전력거래소 연간 파일 한 해 — 그 밖을 API의 넷에 맞춰 묶는다
    이름 = [f for f in os.listdir(os.path.join(RAW, "kpx")) if f"{해}1231" in f][0]
    d = pd.read_csv(os.path.join(RAW, "kpx", 이름), encoding="utf-8-sig")
    d.columns = ["date", "hour", "fuel", "cap_mw", "mwh"]                              # 준비 ④와 같다
    d["fuel"] = d["fuel"].str.strip()
    d = d[~d["fuel"].isin(["LNG", "원자력"] + 석탄종류)].copy()                        # LNG · 원전 · 석탄을 뺀 나머지가 그 밖 — LNG 줄은 여기서 버린다
    d["묶음"] = d["fuel"].map({"수력": "수력", "양수": "양수", "중유": "유류", "경유": "유류"}).fillna("신재생 · 기타")   # 나머지는 모두 신재생 · 기타
    d["ts"] = pd.to_datetime(d["date"]) + pd.to_timedelta(d["hour"].astype(int) - 1, unit="h")   # 1 ~ 24시를 0 ~ 23시 시각으로
    return d.pivot_table(index="ts", columns="묶음", values="mwh", aggfunc="sum").fillna(0)
넷 = pd.concat([연간넷(2023), a24.rename(columns=바꿈)[넷이름], 연간넷(2025)]).sort_index().reindex(전체.index)[넷이름]   # 2023 ~ 2025년 시각마다 넷
넷26 = a26.rename(columns=바꿈)[넷이름]                                              # 2026년 — 준비 ④에서 이미 2026년 시각에 맞춰 두었다
assert 넷.notna().all().all() and 넷26.notna().all().all(), "빈 시각이 있다"

GWh = lambda mwh: mwh * 720 / 1000                                                   # 한 시각 평균(MWh) × 720시각 ÷ 1,000 = 30일 기준(GWh)
차이넷 = pd.DataFrame({k: GWh((넷26[k] - 평소라면값(넷[k])).groupby(넷26.index.month).mean()) for k in 넷이름})   # 실제 − 모델 예측, 달마다
차이넷["넷을 더한 값"] = 차이넷[넷이름].sum(axis=1)                                  # 2-마의 「그 밖」과 같아야 한다
어긋남 = (차이넷["넷을 더한 값"] - (-GWh(달몫["그 밖"]))).abs().max()                # 2-마 「그 밖」(준비 ④의 달몫은 부호가 반대)과 비교

쉼표 = lambda v: f"{v:+,.0f}"                                                        # 천 단위 쉼표, + · − 를 붙여서
표1 = 차이넷.apply(lambda s: s.map(쉼표))
표1.index = [f"{m}월" for m in 표1.index]                                            # 1 → 「1월」
print("[표 1] 2026년 달마다 「그 밖」 발전원별 발전량 — 실제 − 모델 예측, 30일 기준 GWh · + 면 실제가 모델 예측보다 많았다, − 면 적었다 (5월은 15일 평균)")
print("       모델 예측 = 2-마와 같은 선형회귀가 그 발전원 하나만 맞힌 값 · 신재생 · 기타 = 태양광 · 풍력 · 바이오 · 연료전지 · 소수력 같은 것(태양광만 따로는 없다)")
print()
display(표1)
print(f"넷을 더한 값과 2-마의 「그 밖」이 가장 크게 어긋난 달 {어긋남:.1f} GWh (0이어야 한다)")

[표 1] 2026년 달마다 「그 밖」 발전원별 발전량 — 실제 − 모델 예측, 30일 기준 GWh · + 면 실제가 모델 예측보다 많았다, − 면 적었다 (5월은 15일 평균)
       모델 예측 = 2-마와 같은 선형회귀가 그 발전원 하나만 맞힌 값 · 신재생 · 기타 = 태양광 · 풍력 · 바이오 · 연료전지 · 소수력 같은 것(태양광만 따로는 없다)



,수력,양수,유류,신재생 · 기타,넷을 더한 값
1월,-9,-68,-52,+448,+319
2월,-5,-71,-39,+396,+282
3월,-27,-123,-32,+166,-15
4월,-103,-122,-19,+247,+4
5월,-112,-107,-15,-374,-608
6월,-97,-35,-11,-348,-491


넷을 더한 값과 2-마의 「그 밖」이 가장 크게 어긋난 달 0.0 GWh (0이어야 한다)


## 2-사

**어떻게 뽑나** — 밤에는 태양광이 0이라, 그날 1 ~ 5시 신재생 · 기타의 평균을 태양광이 아닌 것으로 보고, 시각마다 「신재생 · 기타 − 그날 1 ~ 5시 평균」을 태양광으로 어림합니다(0보다 작으면 0). 2024년 실제와 맞춰 보면 3 ~ 10월은 −0.7 ~ +7.8%, 11 ~ 2월은 +8.3 ~ +13.0% 크게 나왔습니다(9단계 1-바). 모델 예측은 2023 ~ 2025년의 실제 태양광 발전량을 2-마와 같은 선형회귀로 배워 맞힌 값입니다.

**표의 숫자** — 모든 칸은 「실제 − 모델 예측」(+ 면 실제가 모델 예측보다 많았다, − 면 적었다)이고, 30일 기준 GWh입니다(5월은 15일 평균). 2026년 태양광의 「실제」는 역산 어림입니다. 「나머지」는 신재생 · 기타에서 태양광을 뺀 값이라, 태양광 + 나머지 = 신재생 · 기타입니다. 모델끼리 비교하는 표가 아닙니다.

- 5 · 6월 신재생 · 기타의 − 가 대부분 태양광이면 — 태양광 설비가 늘었는데도 모델 예측(서울 일사를 넣고 맞힌 값)보다 덜 만든 것입니다. 서울 밖 날씨 때문인지 출력 제어 때문인지는 이 프로젝트 자료로 구별할 수 없습니다.
- 대부분 나머지면 — 풍력 · 바이오 · 연료전지 같은 것 가운데 무엇인지는 2026년 자료로 더 나눌 수 없습니다.

In [159]:
# 10단계 결과 해석 · 2-사
# ── 신재생 · 기타를 태양광(역산 어림)과 나머지로 나눈다 — 실제 − 모델 예측, 30일 기준 GWh ──
신재26 = 넷26["신재생 · 기타"]                                                       # 2026년 신재생 · 기타 — 2-바와 같다
밤시각 = 신재26.index.hour.isin([0, 1, 2, 3, 4])                                     # 1 ~ 5시(0 ~ 4시 시각) — 태양광이 0인 때
밤 = 신재26[밤시각].groupby(신재26.index[밤시각].normalize()).mean()                 # 그날 밤 평균 — 9단계 1-바 · 2-나와 같다
태양광26 = (신재26 - 밤.reindex(신재26.index.normalize()).values).clip(lower=0)     # 신재생 · 기타 − 그날 밤 평균, 0보다 작으면 0 — 역산 어림
태양광실제 = 전체["solar_mwh"]                                                       # 2023 ~ 2025년은 실제 태양광 발전량이 있다(9단계 2-나와 같다)
assert 태양광실제.notna().all() and 태양광26.notna().all(), "빈 시각이 있다"
태양광차이 = GWh((태양광26 - 평소라면값(태양광실제)).groupby(태양광26.index.month).mean())   # 실제(역산 어림) − 모델 예측, 달마다

표 = pd.DataFrame({"신재생 · 기타 (2-바)": 차이넷["신재생 · 기타"], "그 가운데 태양광 (역산 어림)": 태양광차이})
표["태양광을 뺀 나머지 (풍력 · 바이오 · 연료전지 같은 것)"] = 표["신재생 · 기타 (2-바)"] - 표["그 가운데 태양광 (역산 어림)"]   # 태양광 + 나머지 = 신재생 · 기타
표1 = 표.apply(lambda s: s.map(쉼표))
표1.index = [f"{m}월" for m in 표1.index]                                            # 1 → 「1월」
print("[표 1] 2026년 달마다 신재생 · 기타를 태양광과 나머지로 나눔 — 실제 − 모델 예측, 30일 기준 GWh · + 면 실제가 모델 예측보다 많았다, − 면 적었다 (5월은 15일 평균)")
print("       2026년 태양광 = 신재생 · 기타 − 그날 1 ~ 5시 평균(역산 어림 — 9단계 1-바) · 모델 예측 = 2023 ~ 2025년 실제 태양광을 2-마와 같은 선형회귀로 배워 맞힌 값")
print()
display(표1)

[표 1] 2026년 달마다 신재생 · 기타를 태양광과 나머지로 나눔 — 실제 − 모델 예측, 30일 기준 GWh · + 면 실제가 모델 예측보다 많았다, − 면 적었다 (5월은 15일 평균)
       2026년 태양광 = 신재생 · 기타 − 그날 1 ~ 5시 평균(역산 어림 — 9단계 1-바) · 모델 예측 = 2023 ~ 2025년 실제 태양광을 2-마와 같은 선형회귀로 배워 맞힌 값



,신재생 · 기타 (2-바),그 가운데 태양광 (역산 어림),태양광을 뺀 나머지 (풍력 · 바이오 · 연료전지 같은 것)
1월,+448,+272,+176
2월,+396,+291,+106
3월,+166,+298,-131
4월,+247,+153,+95
5월,-374,-126,-249
6월,-348,-41,-307


## 2-아

**표의 숫자** — [표 1]은 모든 칸이 「실제 − 모델 예측」이고(+ 면 실제가 모델 예측보다 많았다, − 면 적었다), 30일 기준 GWh입니다(5월은 15일 평균). 수요 = 원전 + 그 밖 + 석탄 + LNG이고, 전력시장에서 거래된 모든 발전량의 합입니다. [표 2]는 석탄의 몫 두 가지(%)입니다 — ① 석탄 ÷ 원전과 그 밖이 덜 만든 양(2-마와 같다), ② 석탄 ÷ (석탄 + LNG가 더 만든 양). ②는 수요가 줄어 채울 필요가 없었던 양을 빼고, 더 만든 쪽(석탄과 LNG)만 놓고 석탄의 몫을 본 것입니다. LNG도 모델 예측보다 덜 만든 달은 ②를 내지 않습니다. 모델끼리 비교하는 표가 아닙니다.

이 셀은 검토가 2-마 표의 숫자로 계산한 것을 보고 연 셀이라, 읽는 법도 결과를 짐작한 뒤에 적었습니다.

- 수요가 − 인 달은, 원전과 그 밖이 덜 만든 양 가운데 그만큼은 채울 필요가 없었던 것으로 읽습니다.
- ①과 ②가 크게 다른 달은, 결론 문장에 석탄의 몫을 어느 쪽으로 적을지 정해야 합니다.

In [160]:
# 10단계 결과 해석 · 2-아
# ── 원전 · 그 밖 · 석탄 · LNG와 그 넷을 더한 수요 — 실제 − 모델 예측, 30일 기준 GWh · 석탄의 몫 두 가지 ──
GWh = lambda mwh: mwh * 720 / 1000                                             # 한 시각 평균(MWh) × 720시각 ÷ 1,000 = 30일 기준(GWh)
차 = pd.DataFrame({"원전": -GWh(달몫["원전"]),                                    # 준비 ④의 달몫은 LNG가 늘어난 쪽이 + 라서 부호를 바꾼다 → 실제 − 모델 예측
                   "그 밖": -GWh(달몫["그 밖"]),
                   "석탄": -GWh(달몫["석탄"]),
                   "LNG (선형회귀)": GWh(달몫["선형회귀"])})                       # LNG는 이미 실제 − 모델 예측
차["수요 = 넷을 더한 값"] = 차[["원전", "그 밖", "석탄", "LNG (선형회귀)"]].sum(axis=1)   # 모든 발전원의 합 = 전력 수요(전력시장에서 거래된 양)
어긋남 = (차["수요 = 넷을 더한 값"] - GWh(달몫["수요"])).abs().max()             # 준비 ④가 따로 맞힌 수요와 같아야 한다

덜만든양 = -(차["원전"] + 차["그 밖"])                                            # 원전과 그 밖이 모델 예측보다 덜 만든 양 (2-마의 둘을 더한 값의 크기)
몫1 = 차["석탄"] / 덜만든양 * 100                                                 # 2-마와 같은 비율 — 석탄 ÷ 원전과 그 밖이 덜 만든 양
더만든양 = 차["석탄"] + 차["LNG (선형회귀)"]                                      # 석탄과 LNG가 모델 예측보다 더 만든 양
몫2 = (차["석탄"] / 더만든양 * 100).where(차["LNG (선형회귀)"] >= 0)              # 석탄 ÷ (석탄 + LNG) — LNG도 덜 만든 달은 내지 않는다

쉼표 = lambda v: f"{v:+,.0f}"                                                     # 천 단위 쉼표, + · − 를 붙여서
표1 = 차.apply(lambda s: s.map(쉼표))
표1.index = [f"{m}월" for m in 표1.index]
표2 = pd.DataFrame({"① 석탄 ÷ 원전과 그 밖이 덜 만든 양 (2-마)": 몫1.map(lambda v: f"{v:.0f}%"),
                    "② 석탄 ÷ (석탄 + LNG가 더 만든 양)": 몫2.map(lambda v: "— (LNG도 덜 만들었다)" if pd.isna(v) else f"{v:.0f}%")})
표2.index = 표1.index

print("[표 1] 2026년 달마다 발전량 — 실제 − 모델 예측, 30일 기준 GWh · + 면 실제가 모델 예측보다 많았다, − 면 적었다 (5월은 15일 평균)")
print("       수요 = 원전 + 그 밖 + 석탄 + LNG · 수요가 − 인 달은 모든 발전원을 합친 양이 모델 예측보다 적었다")
display(표1)
print(f"수요 칸과 준비 ④가 따로 맞힌 수요가 가장 크게 어긋난 달 {어긋남:.1f} GWh (0이어야 한다)")
print()
print("[표 2] 석탄의 몫 두 가지 (%) — ①은 2-마와 같고, ②는 수요가 줄어 채울 필요가 없었던 양을 빼고 석탄과 LNG만 놓고 본 것")
display(표2)

[표 1] 2026년 달마다 발전량 — 실제 − 모델 예측, 30일 기준 GWh · + 면 실제가 모델 예측보다 많았다, − 면 적었다 (5월은 15일 평균)
       수요 = 원전 + 그 밖 + 석탄 + LNG · 수요가 − 인 달은 모든 발전원을 합친 양이 모델 예측보다 적었다


,원전,그 밖,석탄,LNG (선형회귀),수요 = 넷을 더한 값
1월,"-1,551",+319,+5,+809,-419
2월,"-2,625",+282,+561,"+1,008",-774
3월,"-2,749",-15,+804,+746,"-1,214"
4월,"-1,059",+4,+238,+235,-582
5월,"-1,618",-608,"+2,039",+194,+6
6월,-500,-491,"+1,072","-1,235","-1,154"


수요 칸과 준비 ④가 따로 맞힌 수요가 가장 크게 어긋난 달 0.0 GWh (0이어야 한다)

[표 2] 석탄의 몫 두 가지 (%) — ①은 2-마와 같고, ②는 수요가 줄어 채울 필요가 없었던 양을 빼고 석탄과 LNG만 놓고 본 것


,① 석탄 ÷ 원전과 그 밖이 덜 만든 양 (2-마),② 석탄 ÷ (석탄 + LNG가 더 만든 양)
1월,0%,1%
2월,24%,36%
3월,29%,52%
4월,23%,50%
5월,92%,91%
6월,108%,— (LNG도 덜 만들었다)


# 물음 3 — 이 프로젝트가 본 것과 보지 못한 것의 경계는 어디인가

> **결정** — LNG가 들어오고 쌓이고 쓰이는 양 가운데 이 프로젝트가 본 것은 LNG 발전량 하나다. 2026년에 모델 예측과 달랐던 원전 · 석탄 · 전력 수요와 그 까닭 후보는 모두 모델 밖에 있고, 재고와 들여온 양은 보지 못해 LNG가 모자랐는지는 말할 수 없다